# Phase 5: Legal Statutes Prediction

 _This is submission 3 where we fine tune our phase 1 Long-Context Decoder-only transformer with QLoRA adaptation for downstream task of Legal Statutes Prediction._

## 1.Login and GPU check

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Dataloading and Pre-processing

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer


HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone" 

# --- 1. Data Loading ---
print("Loading datasets from Hugging Face Hub...")

try:
    # Explicitly specify 'phase_4' here
    hf_dataset = load_dataset(HF_DATASET_REPO, 'phase_5')
    print(f"Successfully loaded splits: {list(hf_dataset.keys())}")
except Exception as e:
    print(f"Error loading dataset: {e}")

def standardize_labels(example, is_test=False):
    if not is_test:
        if 'pred' in example and 'label' not in example:
             example['label'] = example['pred']
    return example

# Apply standard formatting
for split in hf_dataset.keys():
    is_test_split = (split == 'test')
    hf_dataset[split] = hf_dataset[split].map(lambda x: standardize_labels(x, is_test=is_test_split))

# Assuming you have N total distinct statutes in Phase 5
NUM_STATUTES = 100 

def multi_hot_encode(example):
    # Convert list of statute IDs to a binary vector of size NUM_STATUTES
    labels = np.zeros(NUM_STATUTES, dtype=np.float32)
    if "labels" in example and example["labels"] is not None:
        for statute_id in example["labels"]:
            labels[statute_id] = 1.0
        example["labels"] = labels
    return example

hf_dataset = hf_dataset.map(multi_hot_encode)
print("Data loading complete.")

In [ ]:
# --- 2. Preprocessing & Strategic Truncation ---

from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

BPE_FILE = "/kaggle/input/datasets/vaibhavim081203/phase3feed/cs781_phase1/legal_bpe.json"

# Load backend tokenizer directly
backend_tok = Tokenizer.from_file(BPE_FILE)

# Wrap into Hugging Face PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend_tok)

# Define special tokens according to your Phase 1 setup
tokenizer.pad_token = "[PAD]" if "[PAD]" in tokenizer.get_vocab() else tokenizer.eos_token
tokenizer.cls_token = "[CLS]" if "[CLS]" in tokenizer.get_vocab() else "<s>"
tokenizer.sep_token = "[SEP]" if "[SEP]" in tokenizer.get_vocab() else "</s>"

print(f"Tokenizer loaded successfully! Vocab size: {len(tokenizer)}")

# Safe fallback for special tokens
START_ID = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else (tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0)
END_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

MAX_LENGTH = 768 #
STRIDE = 176      # Overlap of 176 tokens to keep sentences intact
MAX_CHUNKS_PER_DOC= 12

def sliding_window_chunking(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        padding=False,
        truncation=False,
        add_special_tokens=False
    )
    
    input_ids_list = []
    attention_mask_list = []
    
    for input_ids in tokenized_inputs["input_ids"]:
        chunks = []
        masks = []
        
        # Slide over the long document
        for i in range(0, len(input_ids), STRIDE):
            chunk = input_ids[i : i + (MAX_LENGTH - 2)]
            # Add Special Tokens safely without injecting None
            final_ids = [START_ID] + chunk + [END_ID]
            attention_mask = [1] * len(final_ids)
            
            chunks.append(final_ids)
            masks.append(attention_mask)
            
        # Enforce the cap safely using list slicing
        # If len(chunks) <= 12, it does nothing. If > 12, it truncates the tail.
        input_ids_list.append(chunks[:MAX_CHUNKS_PER_DOC])
        attention_mask_list.append(masks[:MAX_CHUNKS_PER_DOC])
        
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list
    }

tokenized_datasets = hf_dataset.map(sliding_window_chunking, batched=True, remove_columns=["text"])

In [ ]:
import torch
import numpy as np

print("Calculating class frequencies for pos_weight...")

# Total number of training cases
num_train_cases = len(hf_dataset["train"])
statute_counts = np.zeros(NUM_STATUTES)

# Count positive occurrences of each statute
for example in hf_dataset["train"]:
    if "pred" in example and example["pred"] is not None:
        for statute_id in example["pred"]:
            statute_counts[statute_id] += 1

# Formula for BCE pos_weight: (number of negative samples) / (number of positive samples)
# We add a small epsilon to prevent division by zero for statutes that might have 0 counts
epsilon = 1e-5
pos_weights = (num_train_cases - statute_counts) / (statute_counts + epsilon)

# PRO TIP: Clamp the maximum weight. 
# If a statute appears only 1 time in 30,000 cases, its weight would be 29,999. 
# This will cause explosive gradients (NaN loss). Capping it at ~50.0 is much safer.
pos_weights = np.clip(pos_weights, a_min=1.0, a_max=50.0)

# Convert to a PyTorch tensor
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)
print("pos_weight tensor created successfully!")

## 3. Model Initialization and LORMA adaptations

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

NEG_INF = -65504.0   # most negative finite fp16 value: "masked" without risking NaN from -inf arithmetic


@dataclass
class LongLMConfig:
    vocab_size: int = 32000      # Custom BPE tokenizer vocabulary
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 10           # ~95M parameters (same size as the dense decoder -> fair comparison)
    max_seq_len: int = 4096      # trained context length (ALiBi lets it run longer at test time)
    window: int = 256            # local block size W: a query sees its own block + the previous block (W..2W-1 tokens)
    global_stride: int = 64      # every S-th token (positions S-1, 2S-1, ...) is a GLOBAL token visible to all later tokens
    dropout: float = 0.1
    bias: bool = True


def compute_alibi_slopes(n_heads: int) -> torch.Tensor:
    """Geometric ALiBi slopes (Press et al.), reference-exact also for head counts that are not a power of two."""
    n = 2 ** math.floor(math.log2(n_heads))
    base = 2 ** (-(2 ** -(math.log2(n) - 3)))
    slopes = [base * (base ** i) for i in range(n)]
    if n < n_heads:
        extra_base = 2 ** (-(2 ** -(math.log2(2 * n) - 3)))
        slopes.extend([extra_base ** (2 * i + 1) for i in range(n_heads - n)])
    return torch.tensor(slopes, dtype=torch.float32)


class LongContextAttention(nn.Module):
    """
    Causal sparse attention = sliding local window + strided global tokens, with ALiBi distance penalties.

    Rule for a query at position i (block b = i // W):
      local : keys j with  (b-1)*W <= j <= i                  (own block + previous block)
      global: keys j with  j % S == S-1  and  j < (b-1)*W     (older "summary" tokens every S positions)
    Training / prefill compute this block-wise in O(T * (2W + T/S)) instead of O(T^2); decoding gathers just the
    allowed keys of the KV cache, so every new token costs O(2W + T/S).
    Heads with small ALiBi slopes keep global tokens relevant over thousands of tokens (long-range heads);
    heads with large slopes focus on the local window.
    """
    def __init__(self, config: LongLMConfig):
        super().__init__()
        assert config.d_model % config.n_heads == 0, "d_model must be divisible by n_heads"
        assert config.window % config.global_stride == 0, "window must be a multiple of global_stride"
        self.n_heads = config.n_heads
        self.d_head = config.d_model // config.n_heads
        self.window = config.window
        self.stride = config.global_stride
        self.dropout_p = config.dropout
        self.c_attn = nn.Linear(config.d_model, 3 * config.d_model, bias=config.bias)   # fused Q,K,V
        self.c_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    @staticmethod
    def _bias_for(ctx, dtype):
        """Cast the fp32 ALiBi/mask bias once per dtype and share it between all layers."""
        b = ctx["bias"].get(dtype)
        if b is None:
            b = ctx["bias"][dtype] = ctx["bias"][torch.float32].to(dtype)
        return b

    def _block_attention(self, q, k, v, ctx):
        """q,k,v: (B,H,T,dh)  ->  (B,T,H*dh)"""
        B, H, T, dh = q.shape
        W = self.window
        Tp = ctx["T_pad"]
        if Tp != T:                                    # right-pad to a multiple of W (causal: real tokens unaffected)
            pad = (0, 0, 0, Tp - T)
            q, k, v = F.pad(q, pad), F.pad(k, pad), F.pad(v, pad)
        nb = Tp // W
        qb = q.reshape(B, H, nb, W, dh)
        kb = k.reshape(B, H, nb, W, dh)
        vb = v.reshape(B, H, nb, W, dh)
        zero = kb.new_zeros(B, H, 1, W, dh)            # "previous block" of the first block (fully masked)
        k_loc = torch.cat([torch.cat([zero, kb[:, :, :-1]], dim=2), kb], dim=3)      # (B,H,nb,2W,dh)
        v_loc = torch.cat([torch.cat([zero, vb[:, :, :-1]], dim=2), vb], dim=3)
        gidx = ctx["gidx"]
        Tg = gidx.numel()
        kg = k.index_select(2, gidx).unsqueeze(2).expand(B, H, nb, Tg, dh)           # global keys, shared by all blocks
        vg = v.index_select(2, gidx).unsqueeze(2).expand(B, H, nb, Tg, dh)
        k_all = torch.cat([k_loc, kg], dim=3)                                        # (B,H,nb,K,dh), K = 2W + Tg
        v_all = torch.cat([v_loc, vg], dim=3)
        K = 2 * W + Tg
        # fold (nb, H) into one head axis (nb-major) so the bias (1, nb*H, W, K) broadcasts over the batch for free
        q4 = qb.permute(0, 2, 1, 3, 4).reshape(B, nb * H, W, dh)
        k4 = k_all.permute(0, 2, 1, 3, 4).reshape(B, nb * H, K, dh)
        v4 = v_all.permute(0, 2, 1, 3, 4).reshape(B, nb * H, K, dh)
        y4 = F.scaled_dot_product_attention(
            q4, k4, v4, attn_mask=self._bias_for(ctx, q4.dtype),
            dropout_p=self.dropout_p if self.training else 0.0)                      # (B, nb*H, W, dh)
        y = y4.reshape(B, nb, H, W, dh).permute(0, 1, 3, 2, 4).reshape(B, Tp, H * dh)
        return y[:, :T]

    def forward(self, x, ctx, past_kv=None, use_cache=False):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)   # (B,H,T,dh)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        if ctx["mode"] == "step":                                     # one new token, KV cache holds everything so far
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            present = (k, v) if use_cache else None
            idx = ctx["idx"]
            y = F.scaled_dot_product_attention(
                q, k.index_select(2, idx), v.index_select(2, idx), attn_mask=self._bias_for(ctx, q.dtype))
            y = y.transpose(1, 2).reshape(B, T, C)
        else:                                                         # training / evaluation / prompt prefill
            present = (k, v) if use_cache else None
            y = self._block_attention(q, k, v, ctx)
        return self.resid_dropout(self.c_proj(y)), present


class FeedForward(nn.Module):
    def __init__(self, config: LongLMConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model, bias=config.bias),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: LongLMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = LongContextAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x, ctx, past_kv=None, use_cache=False):
        a, present = self.attn(self.ln1(x), ctx, past_kv, use_cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, present


class CustomLongLM(nn.Module):
    def __init__(self, config: LongLMConfig):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.d_model),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)]),
            ln_f=nn.LayerNorm(config.d_model),
        ))
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight          # weight tying

        self.register_buffer("alibi_slopes", compute_alibi_slopes(config.n_heads).view(1, config.n_heads, 1, 1),
                             persistent=False)
        self._layout_key = None
        self._layout_val = None
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def get_num_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # ------------------------------------------------------------------ attention layouts (built once per forward)
    def _block_layout(self, Tp: int, device):
        """Mask + ALiBi bias for block-wise attention over a (padded) length Tp; cached for repeated identical shapes."""
        key = (Tp, str(device))
        if self._layout_key == key:
            return self._layout_val
        W, S, H = self.config.window, self.config.global_stride, self.config.n_heads
        nb = Tp // W
        gidx = torch.arange(S - 1, Tp, S, device=device)                              # (Tg,) global key positions
        Tg = gidx.numel()
        j = torch.arange(nb, device=device)
        qpos = (j * W).view(nb, 1) + torch.arange(W, device=device).view(1, W)        # (nb,W)
        kloc = ((j - 1) * W).view(nb, 1) + torch.arange(2 * W, device=device).view(1, 2 * W)   # (nb,2W) prev+own block
        kpos = torch.cat([kloc, gidx.view(1, -1).expand(nb, Tg)], dim=1)              # (nb,K)
        dist = qpos.unsqueeze(2) - kpos.unsqueeze(1)                                  # (nb,W,K)
        valid_loc = (kloc >= 0).unsqueeze(1) & (dist[:, :, :2 * W] >= 0)              # (nb,W,2W)
        valid_glob = (gidx.view(1, -1) < ((j - 1) * W).view(nb, 1)).unsqueeze(1).expand(nb, W, Tg)
        valid = torch.cat([valid_loc, valid_glob], dim=2)                             # (nb,W,K)
        bias = -self.alibi_slopes.to(device).float().view(1, H, 1, 1) * dist.clamp(min=0).float().unsqueeze(1)
        bias = bias.masked_fill(~valid.unsqueeze(1), NEG_INF)                         # (nb,H,W,K)
        ctx = {"mode": "block", "T_pad": Tp, "gidx": gidx,
               "bias": {torch.float32: bias.reshape(1, nb * H, W, 2 * W + Tg)}}
        self._layout_key, self._layout_val = key, ctx
        return ctx

    def _step_layout(self, qpos: int, device):
        """Keys the single new token at position `qpos` may attend to (same rule, gathered from the KV cache)."""
        W, S, H = self.config.window, self.config.global_stride, self.config.n_heads
        start = max(0, (qpos // W - 1) * W)
        loc = torch.arange(start, qpos + 1, device=device)
        if start > S - 1:
            glob = torch.arange(S - 1, start, S, device=device)
        else:
            glob = torch.empty(0, dtype=torch.long, device=device)
        idx = torch.cat([glob, loc])
        dist = (qpos - idx).float().view(1, 1, 1, -1)
        bias = -self.alibi_slopes.to(device).float() * dist                           # (1,H,1,K)
        return {"mode": "step", "idx": idx, "bias": {torch.float32: bias}}

    def _run(self, input_ids, past_kvs=None, use_cache=False):
        B, T = input_ids.size()
        W = self.config.window
        if past_kvs is None:
            ctx = self._block_layout(-(-T // W) * W, input_ids.device)
        else:
            assert T == 1, "cached decoding processes one token at a time"
            ctx = self._step_layout(past_kvs[0][0].size(2), input_ids.device)
        x = self.transformer.drop(self.transformer.wte(input_ids))   # ALiBi supplies the position information
        presents = []
        for i, block in enumerate(self.transformer.h):
            x, present = block(x, ctx, None if past_kvs is None else past_kvs[i], use_cache)
            presents.append(present)
        return self.transformer.ln_f(x), presents

    def forward(self, input_ids, targets=None):
        x, _ = self._run(input_ids)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def token_losses(self, input_ids, targets):
        """Per-token cross-entropy, shape (B,T) - used by the context-length study."""
        x, _ = self._run(input_ids)
        logits = self.lm_head(x)
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                               reduction="none").view(targets.shape)

    def forward_cached(self, input_ids, past_kvs=None):
        """Prefill (past_kvs=None, any length) or one-token step. Returns (last-position logits (B,1,V), KV cache)."""
        x, presents = self._run(input_ids, past_kvs, use_cache=True)
        return self.lm_head(x[:, [-1], :]), presents

    @staticmethod
    def _sample(logits, temperature, top_p):
        if temperature is None or temperature <= 0:
            return logits.argmax(dim=-1, keepdim=True)
        probs = F.softmax(logits.float() / temperature, dim=-1)
        if top_p is not None and top_p < 1.0:
            sorted_p, sorted_i = torch.sort(probs, descending=True, dim=-1)
            cum = sorted_p.cumsum(dim=-1)
            sorted_p = sorted_p.masked_fill((cum - sorted_p) > top_p, 0.0)
            sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True)
            return sorted_i.gather(-1, torch.multinomial(sorted_p, 1))
        return torch.multinomial(probs, 1)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=1.0, top_p=1.0, stop_fn=None, check_every=16):
        """KV-cached generation: one block-wise prefill, then one token per step (each step touches only 2W + T/S keys)."""
        self.eval()
        room = max(1, self.config.max_seq_len - max_new_tokens)
        input_ids = input_ids[:, -room:]                      # keep the END of an over-long prompt
        logits, past = self.forward_cached(input_ids, None)
        new_ids = []
        for t in range(max_new_tokens):
            nxt = self._sample(logits[:, -1, :], temperature, top_p)
            new_ids.append(nxt)
            if stop_fn is not None and (t + 1) % check_every == 0 and stop_fn(torch.cat(new_ids, 1)[0].tolist()):
                break
            if t + 1 < max_new_tokens:
                logits, past = self.forward_cached(nxt, past)
        return torch.cat(new_ids, dim=1)

In [ ]:
!pip install bitsandbytes

In [ ]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb
# from model_lc import CustomLongLM, LongLMConfig # Assuming this is accessible

class QLoRALinear(nn.Module):
    """Custom 4-bit Quantized Linear Layer with LoRA adapters."""
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=True):
        super().__init__()
        # The frozen 4-bit base weight
        self.base_layer = bnb.nn.Linear4bit(
            in_features, out_features, bias=bias,
            compute_dtype=torch.float16,
            quant_type="nf4"
        )
        
        # The trainable LoRA parameters
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(in_features, r))
        self.lora_B = nn.Parameter(torch.zeros(r, out_features))
        
        nn.init.normal_(self.lora_A, std=1 / in_features)
        nn.init.zeros_(self.lora_B)
        
    def forward(self, x):
        base_out = self.base_layer(x)
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scaling
        return base_out + lora_out

def inject_qlora_into_sparse_model(model, r=8, alpha=16):
    """Replaces standard linear layers in the transformer with QLoRA layers."""
    for block in model.transformer.h:
        # Inject into fused Attention QKV
        old_attn = block.attn.c_attn
        new_attn = QLoRALinear(old_attn.in_features, old_attn.out_features, r=r, alpha=alpha, bias=True)
        # (In a real scenario, you'd copy old_attn.weight into new_attn.base_layer.weight here)
        block.attn.c_attn = new_attn
        
        # Inject into Attention Projection
        old_proj = block.attn.c_proj
        new_proj = QLoRALinear(old_proj.in_features, old_proj.out_features, r=r, alpha=alpha, bias=True)
        block.attn.c_proj = new_proj
        
    # Freeze everything EXCEPT LoRA parameters
    for name, param in model.named_parameters():
        if 'lora_' not in name:
            param.requires_grad = False
    return model

class HierarchicalSparseForMultiLabel(nn.Module):
    def __init__(self, config, num_labels=NUM_STATUTES, pos_weight_tensor=None):
        super().__init__()
        # 1. Base Model (Phase 1 Sparse Decoder)
        self.base_model = CustomLongLM(config)
        
        # Sever weight tying for Multi-GPU safety
        if hasattr(self.base_model, 'lm_head'):
            clean_weight = self.base_model.transformer.wte.weight.clone().detach()
            self.base_model.transformer.wte.weight = nn.Parameter(clean_weight)
            del self.base_model.lm_head
            
        # 2. Inject QLoRA
        self.base_model = inject_qlora_into_sparse_model(self.base_model, r=8, alpha=16)
        
        # 3. Global Attention Pooling Layer (Trainable)
        self.chunk_attention = nn.Linear(config.d_model, 1)
        
        # 4. Final Classification Head (Trainable)
        self.classifier = nn.Linear(config.d_model, num_labels)
        
        if pos_weight_tensor is not None:
            self.register_buffer('pos_weight', pos_weight_tensor)
        else:
            self.pos_weight = None

    def forward(self, input_ids, attention_mask, chunk_mask, labels=None):
        batch_size, num_chunks, seq_len = input_ids.shape
        
        # --- STAGE A: Local Sparse Decoding ---
        flat_input_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_attention_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        
        # Pass through Sparse Transformer
        hidden_states, _ = self.base_model._run(flat_input_ids) # (B*num_chunks, 768, d_model)
        
        # Mean-pool unmasked tokens to get chunk representations
        mask_expanded = flat_attention_mask.unsqueeze(-1).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        chunk_reps = (sum_embeddings / sum_mask).view(batch_size, num_chunks, -1) # (B, num_chunks, d_model)
        
        # --- STAGE B: Global Attention Pooling ---
        # Score each chunk
        attn_logits = self.chunk_attention(chunk_reps).squeeze(-1) # (B, num_chunks)
        
        # Mask out completely empty chunks (preventing them from getting attention)
        min_val = torch.finfo(attn_logits.dtype).min
        attn_logits = attn_logits.masked_fill(~chunk_mask, min_val)
        
        # Convert scores to probabilities
        chunk_probs = F.softmax(attn_logits, dim=1) # (B, num_chunks)
        
        # Weighted sum of chunks to create final document vector
        doc_representation = torch.sum(chunk_reps * chunk_probs.unsqueeze(-1), dim=1) # (B, d_model)
        
        # --- STAGE C: Classification ---
        logits = self.classifier(doc_representation)
        
        loss = None
        if labels is not None:
            loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=self.pos_weight)
            
        return {"logits": logits, "loss": loss, "chunk_probs": chunk_probs}

# Initialize (Make sure pos_weight_tensor is created as before)
config = LongLMConfig(vocab_size=len(tokenizer), max_seq_len=MAX_LENGTH) # Local window config
model = HierarchicalSparseForMultiLabel(config, pos_weight_tensor=pos_weight_tensor)

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")

## 4. Training loop and Evaluation

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW  # <-- Import from PyTorch instead
from tqdm.auto import tqdm
import random
import numpy as np
import os
import gc
# --- 1. DataLoader & Padding Setup ---

CHUNK_LEN = 768
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

def collate_fn_hierarchical(batch):
    max_chunks = max(len(item['input_ids']) for item in batch)
    
    batch_input_ids = []
    batch_attention_mask = []
    batch_chunk_mask = [] # Tracks which chunks are real vs. padding
    labels, ids = [], []
    
    for item in batch:
        chunks = item['input_ids']
        masks = item['attention_mask']
        
        padded_chunks = []
        padded_masks = []
        chunk_mask = []
        
        # Process existing chunks
        for chunk, mask in zip(chunks, masks):
            pad_len = CHUNK_LEN - len(chunk)
            padded_chunks.append(chunk + [PAD_TOKEN_ID] * pad_len)
            padded_masks.append(mask + [0] * pad_len)
            chunk_mask.append(1) # Real chunk
            
        # Pad missing chunks up to max_chunks
        while len(padded_chunks) < max_chunks:
            padded_chunks.append([PAD_TOKEN_ID] * CHUNK_LEN)
            padded_masks.append([0] * CHUNK_LEN)
            chunk_mask.append(0) # Empty padding chunk
            
        batch_input_ids.append(padded_chunks)
        batch_attention_mask.append(padded_masks)
        batch_chunk_mask.append(chunk_mask)
        
        # --- THE FIX ---
        # Check that the key exists AND that it is not None
        if 'labels' in item and item['labels'] is not None:
            labels.append(item['labels'])
        if 'id' in item:
            ids.append(item['id'])
            
    return {
        'input_ids': torch.tensor(batch_input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(batch_attention_mask, dtype=torch.long),
        'chunk_mask': torch.tensor(batch_chunk_mask, dtype=torch.bool),
        'labels': torch.tensor(labels, dtype=torch.float32) if labels else None,
        'ids': ids
    }

# Create DataLoaders (adjust batch_size based on your GPU VRAM, e.g., 4, 8, or 16)
BATCH_SIZE = 2
train_loader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=collate_fn_hierarchical)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=BATCH_SIZE, collate_fn=collate_fn_hierarchical)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=BATCH_SIZE, collate_fn=collate_fn_hierarchical)


# --- 2. Training Setup ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)


LEARNING_RATE = 3e-4 # Higher LR is typical for PEFT compared to full fine-tuning

EPOCHS = 3
# How many epochs to actually run in this Kaggle session (set to 1 to beat the 12h limit)
EPOCHS_THIS_SESSION = 1

# Define total steps for the FULL 3-epoch journey so the LR decays correctly
total_steps = len(train_loader) * EPOCHS

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * total_steps), 
    num_training_steps=total_steps
)
scaler = torch.amp.GradScaler('cuda')
# Start tracking variables
start_epoch = 2
best_macro_f1 = 0.0
best_threshhold= 0.0
# --- 3. Training & Evaluation Loop ---

def evaluate_hierarchical(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            with torch.amp.autocast('cuda'):
                outputs = model(
                    input_ids=batch['input_ids'].to(device), 
                    attention_mask=batch['attention_mask'].to(device),
                    chunk_mask=batch['chunk_mask'].to(device)
                )
                probs = torch.sigmoid(outputs['logits'])
            
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch['labels'].numpy())
            
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        f1 = f1_score(all_labels, (all_preds > thresh).astype(int), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    return best_f1, best_thresh
 



ACCUMULATION_STEPS = 8 # physical batch (2) * steps (4) = effective batch (8)

checkpoint_path = "/kaggle/input/notebooks/vaibhavim081203/cs781p5s3/lc_checkpoint.pt"

if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    # weights_only=False required in PyTorch 2.6 to load numpy/random states
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if hasattr(model, 'module'):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
        
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_macro_f1 = checkpoint['best_macro_f1']
    
    # Safe RNG Restoration (.cpu().byte() prevents the TypeError)
    torch.set_rng_state(checkpoint['torch_rng_state'].cpu().byte())
    cuda_states = [state.cpu().byte() for state in checkpoint['cuda_rng_state']]
    torch.cuda.set_rng_state_all(cuda_states)
    np.random.set_state(checkpoint['numpy_rng_state'])
    random.setstate(checkpoint['python_rng_state'])
    print(f"Resumed successfully from Epoch {start_epoch}")
else:
    print("No checkpoint found. Starting training from scratch.")

print("Starting Hierarchical QLoRA Training...")
for epoch in range(start_epoch,start_epoch+ EPOCHS_THIS_SESSION):
    model.train()
    total_loss = 0
    optimizer.zero_grad() 
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for step, batch in enumerate(progress_bar):
        with torch.amp.autocast('cuda'):
            outputs = model(
                input_ids=batch['input_ids'].to(device), 
                attention_mask=batch['attention_mask'].to(device),
                chunk_mask=batch['chunk_mask'].to(device),
                labels=batch['labels'].to(device)
            )
            loss = outputs['loss']
            
            if loss.dim() > 0: # Multi-GPU safety
                loss = loss.mean()
            loss = loss / ACCUMULATION_STEPS
            
        scaler.scale(loss).backward()
        
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad() 
            
        actual_loss = loss.item() * ACCUMULATION_STEPS
        total_loss += actual_loss
        progress_bar.set_postfix({'loss': actual_loss})
            
        # --- THE OPTIMIZED MEMORY FIX ---
        # Only trigger the heavy garbage collection every 200 steps
        if (step + 1) % (ACCUMULATION_STEPS * 400) == 0:
            # 1. Manually delete the heavy intermediate tensors from the current step
            del outputs, loss
            
            # 2. Force Python to run garbage collection
            gc.collect()
            
            # 3. Force PyTorch to release the fragmented VRAM
            torch.cuda.empty_cache()

     # Save Bulletproof Checkpoint
    model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    torch.save({
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_macro_f1': best_macro_f1,
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state_all(),
        'numpy_rng_state': np.random.get_state(),
        'python_rng_state': random.getstate()
    }, "lc_checkpoint.pt")
        
    val_f1, best_threshold = evaluate_hierarchical(model, val_loader)
    print(f"Epoch {epoch+1} - Train Loss: {total_loss/len(train_loader):.4f} - Dev F1: {val_f1:.4f} (Thresh: {best_threshold:.2f})")
    
    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(model_state, "best_hierarchical_qlora.pt")
                
        # --- ADD THIS LINE: Save the threshold for the best epoch ---
        torch.save({'threshold': best_threshold}, "best_threshold.pt")


## 5. Prediction for Test cases and submission

In [ ]:
import json
import zipfile
import torch
from tqdm.auto import tqdm

print("Generating predictions for the test set...")

# 1. Load best model weights safely
best_weights = torch.load("best_hierarchical_qlora.pt", map_location=device, weights_only=True)
model.load_state_dict(best_weights,strict=False)

model.eval()


# 2. Load the optimal threshold safely from the best epoch
try:
    best_t = torch.load("best_threshold.pt", weights_only=True)['threshold']
    print(f"Loaded optimal threshold: {best_t}")
except FileNotFoundError:
    best_t = 0.50 # Safe fallback
    print(f"Using fallback threshold: {best_t}")



predictions = []

# 3. Run inference
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)      
        
        # Use AMP for inference speed and safety
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs['logits']
            
            # Apply Sigmoid for MULTI-LABEL probabilities
            probs = torch.sigmoid(logits)
        
        # Apply the threshold to create a binary multi-hot vector (0s and 1s)
        binary_preds = (probs > best_t).int().cpu().tolist()
        
        # Extract the case IDs 
        batch_ids = batch['ids']
        
        # Now 'pred' is a list of 0s and 1s for all NUM_STATUTES
        for case_id, pred in zip(batch_ids, binary_preds):
            predictions.append({
                "id": case_id,
                "pred": pred 
            })

# 4. Save to prediction.json
output_file = "prediction.json"
with open(output_file, "w", encoding='utf-8') as f:
    json.dump(predictions, f, indent=4)

print(f"Predictions saved to {output_file}.")

# 5. Zip the file for Codabench submission
zip_filename = "submission.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # arcname ensures the file sits at the root of the zip
    zipf.write(output_file, arcname=output_file)

print(f"Successfully created {zip_filename}! You are ready to submit to Codabench.")